# 03 · Spatial upscaling — our key result, reproduced live

*Runnable notebook. We reproduce the headline finding from scratch: predicting ET at an
**unmonitored** tower fails with 5 similar Everglades sites but works once the training set
spans 13 diverse coastal wetlands. Set the kernel to **Python (coastal-et)** and Run All.*

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Derive the project root without hardcoding anyone's personal path:
#   env override -> parent of the notebook's folder -> shared fallback.
ROOT = os.environ.get("COASTAL_ET_ROOT")
if not ROOT or not os.path.isdir(os.path.join(ROOT, "data", "processed")):
    cand = os.path.dirname(os.getcwd())                 # notebook lives in <ROOT>/notebooks
    ROOT = cand if os.path.isdir(os.path.join(cand, "data", "processed")) \
        else "/anvil/projects/x-ees260113/team2/coastal-et"
PROC = f"{ROOT}/data/processed"
FIG = f"{ROOT}/figures"; os.makedirs(FIG, exist_ok=True)
print("project root:", ROOT)

# Nature-ish figure defaults
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 8, "axes.linewidth": 0.7, "figure.dpi": 120,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7})
INK = "#1a1a1a"

In [ ]:
# The 14 predictors: 7 satellite + 7 meteorology. Target is measured closed ET.
SAT = ["LAI", "EVI2", "SAVI", "NDVI", "NDWI", "MNDWI", "LST_K"]
MET = ["TA_ERA", "VPD_ERA", "SW_IN_ERA", "WS_ERA", "ETo_mm", "DOY_sin", "DOY_cos"]
FEATS = SAT + MET
EVERGLADES = ["US-Esm", "US-TaS", "US-Skr", "US-Elm", "US-EvM"]

d = pd.read_parquet(f"{PROC}/more_sites_table.parquet")
SITES = sorted(d.SITE_ID.unique())
print(f"{len(d)} overpass matches | {len(SITES)} sites | {len(FEATS)} features")
print("target: ET_closed_mm (measured, closure-corrected daily ET)")
d[["SITE_ID", "year"] + FEATS + ["ET_closed_mm"]].head()

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_absolute_error

def evaluate(make_model, data, scheme, feats=FEATS):
    """Return (R2, MAE, y_true, y_pred) for one CV scheme.
    kfold=predict at monitored sites; year=predict unseen years;
    site=predict a completely unseen tower (spatial upscaling)."""
    yt, yp = [], []
    if scheme == "kfold":
        for tri, tei in KFold(10, shuffle=True, random_state=0).split(data):
            m = clone(make_model()).fit(data.iloc[tri][feats].values, data.iloc[tri].ET_closed_mm.values)
            yp.append(m.predict(data.iloc[tei][feats].values)); yt.append(data.iloc[tei].ET_closed_mm.values)
    elif scheme == "year":
        for s in sorted(data.SITE_ID.unique()):
            ds = data[data.SITE_ID == s]
            for y in sorted(ds.year.dropna().unique()):
                tr, te = ds[ds.year != y], ds[ds.year == y]
                if len(te) < 5 or len(tr) < 15: continue
                m = clone(make_model()).fit(tr[feats].values, tr.ET_closed_mm.values)
                yp.append(m.predict(te[feats].values)); yt.append(te.ET_closed_mm.values)
    else:  # leave-site-out
        for s in sorted(data.SITE_ID.unique()):
            tr, te = data[data.SITE_ID != s], data[data.SITE_ID == s]
            if len(te) < 5: continue
            m = clone(make_model()).fit(tr[feats].values, tr.ET_closed_mm.values)
            yp.append(m.predict(te[feats].values)); yt.append(te.ET_closed_mm.values)
    yt, yp = np.concatenate(yt), np.concatenate(yp)
    return r2_score(yt, yp), mean_absolute_error(yt, yp), yt, yp

print("CV evaluator ready (schemes: kfold, year, site)")

## 5 Everglades sites vs 13 diverse wetlands

We run leave-site-out (predict a fully unseen tower) on the 5-site Everglades subset and
on the full 13-site network, with our best upscaler (ExtraTrees) and the Gaussian process.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODELS = {
    "ExtraTrees": lambda: ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1),
    "GaussProc":  lambda: make_pipeline(StandardScaler(), GaussianProcessRegressor(
                       kernel=ConstantKernel() * RBF() + WhiteKernel(), normalize_y=True, alpha=1e-3, random_state=0)),
}
d5 = d[d.SITE_ID.isin(EVERGLADES)]
res = {}
for mn, mk in MODELS.items():
    for label, data in [("5 Everglades", d5), ("13 wetlands", d)]:
        rk = evaluate(mk, data, "kfold")[0]; ry = evaluate(mk, data, "year")[0]; rs = evaluate(mk, data, "site")[0]
        res[(mn, label)] = (rk, ry, rs)
        print(f"  {mn:<11}{label:<14} kfold={rk:5.2f}  leave-year={ry:5.2f}  leave-site={rs:5.2f}", flush=True)

## The picture: harder tests need more sites

In [ ]:
schemes = ["K-fold\n(monitored)", "Leave-year\n(unseen yr)", "Leave-site\n(unseen tower)"]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True, constrained_layout=True)
x = np.arange(3); w = 0.36
for ax, mn in zip(axes, ["ExtraTrees", "GaussProc"]):
    v5, v13 = res[(mn, "5 Everglades")], res[(mn, "13 wetlands")]
    ax.bar(x - w/2, np.clip(v5, -1.05, 1), w, color="#B0C4DE", label="5 Everglades", zorder=3)
    ax.bar(x + w/2, np.clip(v13, -1.05, 1), w, color="#2C5F8A", label="13 wetlands", zorder=3)
    ax.axhline(0, color="#8a8a8a", lw=0.8)
    for xi, (a, b) in enumerate(zip(v5, v13)):
        ax.text(xi - w/2, max(a, 0)+0.03, f"{a:.2f}", ha="center", fontsize=6, color=INK)
        ax.text(xi + w/2, max(b, 0)+0.03, f"{b:.2f}", ha="center", fontsize=6, color=INK)
    ax.set_title(mn, fontsize=9); ax.set_xticks(x); ax.set_xticklabels(schemes, fontsize=6.8)
    ax.set_ylim(-1.15, 1.0)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
axes[0].set_ylabel("$R^2$"); axes[1].legend(frameon=False, fontsize=7, loc="lower right")
fig.suptitle("Upscaling to an unseen tower needs training-set diversity",
             fontsize=10, fontweight="bold")
fig.savefig(f"{FIG}/cv_comparison_runnable.png", dpi=200, bbox_inches="tight")
plt.show()

## Predicted vs observed at held-out towers (13 sites)

Every point is an overpass at a tower the model never saw in training.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
_, _, yt, yp = evaluate(lambda: ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1), d, "site")
from sklearn.metrics import r2_score, mean_absolute_error
fig, ax = plt.subplots(figsize=(4.0, 4.0), constrained_layout=True)
ax.scatter(yt, yp, s=9, alpha=0.35, color="#2C5F8A", edgecolor="none")
lim = [0, max(yt.max(), yp.max())*1.05]
ax.plot(lim, lim, "--", color="#8a8a8a", lw=0.9)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Observed ET (mm/day)"); ax.set_ylabel("Predicted ET (mm/day)")
ax.text(0.05, 0.92, f"$R^2$={r2_score(yt,yp):.2f}\nMAE={mean_absolute_error(yt,yp):.2f} mm/d",
        transform=ax.transAxes, fontsize=8, va="top")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.set_title("Leave-site-out prediction, 13 wetlands", fontsize=9.5, fontweight="bold")
fig.savefig(f"{FIG}/upscaling_scatter_runnable.png", dpi=200, bbox_inches="tight")
plt.show()

**The finding, reproduced live:** the bottleneck to satellite ET upscaling in coastal
wetlands is **training-set diversity**, not the model or the features. Five spectrally
near-identical Everglades marshes can't teach a transferable relationship; thirteen
wetlands spanning different climates, salinities and canopies can (leave-site $R^2\approx0.7$).